# Problem 2: Case-Sensitive vs Case-Insensitive NER

- Dataset: CoNLL-2003

- Write a Python program to investigate how capitalization affects dictionary-based named entity recognition. Construct an entity dictionary from the training data and perform recognition on the same test sentences using case-sensitive and case-insensitive matching.

- Tasks: Compare the number of entities detected, calculate precision and recall for both approaches, and identify at least five incorrect predictions.

- Expected output: A comparative table showing the performance of both methods.

In [21]:
!pip install -q datasets

from datasets import load_dataset
from collections import Counter, defaultdict
from itertools import product
import math

In [22]:
# ============================================================
# 1. LOAD DATA
# ============================================================

dataset = load_dataset("BramVanroy/conll2003")

train_data = dataset["train"]
validation_data = dataset["validation"]
test_data = dataset["test"].select(range(200))

label_names = train_data.features["ner_tags"].feature.names

ENTITY_MAP = {
    "PER": "PERSON",
    "LOC": "LOCATION",
    "ORG": "ORGANIZATION"
}

ENTITY_TYPES = ["PERSON", "LOCATION", "ORGANIZATION"]
PRIORITY = {name: i for i, name in enumerate(ENTITY_TYPES)}

print("Training sentences:", len(train_data))
print("Validation sentences:", len(validation_data))
print("Test sentences:", len(test_data))

Training sentences: 14041
Validation sentences: 3250
Test sentences: 200


In [23]:
# ============================================================
# 2. EXTRACT ENTITY SPANS
# ============================================================

def extract_entities(tokens, tag_ids):
    entities = []
    current_type = None
    start = None

    def close_entity(end):
        if current_type is not None:
            entities.append((
                start,
                end,
                current_type,
                " ".join(tokens[start:end])
            ))

    for i, tag_id in enumerate(tag_ids):
        tag = label_names[int(tag_id)]
        prefix, _, code = tag.partition("-")
        mapped_type = ENTITY_MAP.get(code)

        if prefix == "B":
            close_entity(i)
            current_type = mapped_type
            start = i if mapped_type else None

        elif prefix == "I":
            if mapped_type == current_type and current_type is not None:
                continue

            # Recover from an invalid I-tag if necessary
            close_entity(i)
            current_type = mapped_type
            start = i if mapped_type else None

        else:
            close_entity(i)
            current_type = None
            start = None

    close_entity(len(tokens))
    return entities

In [24]:
# ============================================================
# 3. BUILD TRAINING DICTIONARIES AND FREQUENCY STATISTICS
# ============================================================

# Exact surface forms, including capitalization
surface_counts = defaultdict(Counter)

# Normalized phrase -> entity-type counts
normalized_counts = defaultdict(Counter)

for example in train_data:
    tokens = example["tokens"]
    tags = example["ner_tags"]

    for start, end, entity_type, phrase in extract_entities(tokens, tags):
        surface = tuple(tokens[start:end])
        normalized = tuple(t.casefold() for t in surface)

        surface_counts[surface][entity_type] += 1
        normalized_counts[normalized][entity_type] += 1

MAX_LEN = max(len(phrase) for phrase in normalized_counts)

print("\nUNIQUE ENTITY DICTIONARY SIZES")

for entity_type in ENTITY_TYPES:
    exact_size = sum(
        1 for counts in surface_counts.values()
        if counts[entity_type] > 0
    )

    normalized_size = sum(
        1 for counts in normalized_counts.values()
        if counts[entity_type] > 0
    )

    print(
        f"{entity_type}: exact={exact_size}, "
        f"normalized={normalized_size}"
    )

print("Maximum dictionary phrase length:", MAX_LEN)


UNIQUE ENTITY DICTIONARY SIZES
PERSON: exact=3614, normalized=3531
LOCATION: exact=1332, normalized=1106
ORGANIZATION: exact=2403, normalized=2312
Maximum dictionary phrase length: 10


In [25]:
# ============================================================
# 4. BASELINE: CASE-SENSITIVE / CASE-INSENSITIVE MATCHING
# ============================================================

def greedy_predict(tokens, case_sensitive=True):
    predictions = []
    i = 0
    n = len(tokens)

    while i < n:
        found = None
        max_length = min(MAX_LEN, n - i)

        for length in range(max_length, 0, -1):
            surface = tuple(tokens[i:i + length])

            if case_sensitive:
                key = surface
                lookup = surface_counts
            else:
                key = tuple(t.casefold() for t in surface)
                lookup = normalized_counts

            type_counts = lookup.get(key)

            if not type_counts:
                continue

            entity_type = min(
                type_counts,
                key=lambda t: (-type_counts[t], PRIORITY[t])
            )

            found = (
                i,
                i + length,
                entity_type,
                " ".join(tokens[i:i + length])
            )
            break

        if found is not None:
            predictions.append(found)
            i = found[1]
        else:
            i += 1

    return predictions

In [26]:
# ============================================================
# 5. GENERATE CANDIDATE SPANS FOR THE TUNED HYBRID
# ============================================================

def generate_candidates(tokens):
    """
    Generate all spans matching the normalized training dictionary.
    Each candidate stores:
      start, end, type, frequency, type confidence,
      exact-case evidence, lowercase flag, surface phrase
    """

    candidates = []
    n = len(tokens)

    for start in range(n):
        for end in range(start + 1, min(n, start + MAX_LEN) + 1):
            surface = tuple(tokens[start:end])
            normalized = tuple(t.casefold() for t in surface)

            counts = normalized_counts.get(normalized)

            if not counts:
                continue

            total = sum(counts.values())
            exact_types = surface_counts.get(surface, Counter())

            has_letters = any(
                character.isalpha()
                for token in surface
                for character in token
            )

            all_lower = (
                has_letters
                and not any(
                    character.isupper()
                    for token in surface
                    for character in token
                )
            )

            for entity_type, count in counts.items():
                confidence = count / total
                exact_case = exact_types.get(entity_type, 0) > 0

                candidates.append((
                    start,
                    end,
                    entity_type,
                    count,
                    confidence,
                    exact_case,
                    all_lower,
                    " ".join(surface)
                ))

    return candidates


# Precompute candidates once instead of rescanning the text
val_tokens = [x["tokens"] for x in validation_data]
test_tokens = [x["tokens"] for x in test_data]

val_candidates = [
    generate_candidates(tokens) for tokens in val_tokens
]

test_candidates = [
    generate_candidates(tokens) for tokens in test_tokens

]

In [27]:
# ============================================================
# 6. SCORE CANDIDATES
# ============================================================

def candidate_score(candidate, params):
    start, end, entity_type, count, confidence, exact, lower, phrase = candidate

    score = (
        math.log1p(count)
        + 1.2 * confidence
        + params["exact_bonus"] * int(exact)
        + 0.15 * (end - start - 1)
    )

    # Lowercase matches are risky unless the same surface form
    # was observed as this entity type in the training data.
    if lower and not exact:
        score -= params["lowercase_penalty"]

    return score

In [28]:
# ============================================================
# 7. SELECT THE BEST NON-OVERLAPPING SPANS
#    USING WEIGHTED INTERVAL DYNAMIC PROGRAMMING
# ============================================================

def hybrid_predict(tokens, candidates, params):
    n = len(tokens)
    ending_at = [[] for _ in range(n + 1)]

    for candidate in candidates:
        start, end, entity_type, count, confidence, exact, lower, phrase = candidate

        if count < params["min_count"]:
            continue

        if confidence < params["min_confidence"]:
            continue

        score = candidate_score(candidate, params)

        if score < params["min_score"]:
            continue

        ending_at[end].append((candidate, score))

    # dp[i] = best total score for the first i tokens
    dp = [0.0] * (n + 1)
    backtrack = [None] * (n + 1)

    for end in range(1, n + 1):
        dp[end] = dp[end - 1]
        best_candidate = None

        for candidate, score in ending_at[end]:
            start = candidate[0]
            total_score = dp[start] + score

            if total_score > dp[end] + 1e-12:
                dp[end] = total_score
                best_candidate = candidate

        backtrack[end] = best_candidate

    # Recover the optimal non-overlapping entity sequence
    predictions = []
    position = n

    while position > 0:
        candidate = backtrack[position]

        if candidate is None:
            position -= 1
            continue

        start, end, entity_type, count, confidence, exact, lower, phrase = candidate

        predictions.append((
            start, end, entity_type, phrase
        ))

        position = start

    predictions.reverse()
    return predictions

In [29]:
# ============================================================
# 8. GOLD LABELS AND METRIC CALCULATION
# ============================================================

def gold_span_set(example):
    return {
        (start, end, entity_type)
        for start, end, entity_type, phrase in extract_entities(
            example["tokens"], example["ner_tags"]
        )
    }


def calculate_metrics(all_predictions, all_gold):
    TP = FP = FN = 0
    detected = 0

    for predictions, gold in zip(all_predictions, all_gold):
        predicted_set = {
            (start, end, entity_type)
            for start, end, entity_type, phrase in predictions
        }

        detected += len(predicted_set)

        TP += len(predicted_set & gold)
        FP += len(predicted_set - gold)
        FN += len(gold - predicted_set)

    precision = TP / (TP + FP) if TP + FP else 0.0
    recall = TP / (TP + FN) if TP + FN else 0.0

    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall else 0.0
    )

    return {
        "Detected": detected,
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "Precision": precision,
        "Recall": recall,
        "F1": f1
    }


val_gold = [gold_span_set(x) for x in validation_data]
test_gold = [gold_span_set(x) for x in test_data]

In [30]:
# ============================================================
# 9. TUNE ON VALIDATION DATA ONLY
# ============================================================

grid = []

for min_count, min_confidence, exact_bonus, lower_penalty, min_score in product(
    [1, 2],
    [0.50, 0.80, 1.00],
    [0.0, 0.8, 1.6],
    [0.0, 1.0, 2.0],
    [1.5, 2.2, 2.9, 3.6]
):
    grid.append({
        "min_count": min_count,
        "min_confidence": min_confidence,
        "exact_bonus": exact_bonus,
        "lowercase_penalty": lower_penalty,
        "min_score": min_score
    })

print("\nTuning", len(grid), "parameter combinations on validation data...")

best_params = None
best_val_metrics = None
best_val_predictions = None
best_key = (-1.0, -1.0, -1.0)

for index, params in enumerate(grid, start=1):
    predictions = [
        hybrid_predict(tokens, candidates, params)
        for tokens, candidates in zip(val_tokens, val_candidates)
    ]

    metrics = calculate_metrics(predictions, val_gold)

    # Maximize F1; use precision and recall as tie-breakers.
    current_key = (
        metrics["F1"],
        metrics["Precision"],
        metrics["Recall"]
    )

    if current_key > best_key:
        best_key = current_key
        best_params = params.copy()
        best_val_metrics = metrics
        best_val_predictions = predictions

print("\nBEST HYBRID PARAMETERS")
for key, value in best_params.items():
    print(f"{key}: {value}")

print("\nBest hybrid validation F1:", round(best_val_metrics["F1"], 4))


Tuning 216 parameter combinations on validation data...

BEST HYBRID PARAMETERS
min_count: 1
min_confidence: 0.5
exact_bonus: 1.6
lowercase_penalty: 1.0
min_score: 3.6

Best hybrid validation F1: 0.6467


In [31]:
# ============================================================
# 10. COMPARE BASELINES AND TUNED HYBRID ON VALIDATION
# ============================================================

val_sensitive = [
    greedy_predict(tokens, True) for tokens in val_tokens
]

val_insensitive = [
    greedy_predict(tokens, False) for tokens in val_tokens
]

validation_methods = {
    "Case-sensitive": val_sensitive,
    "Case-insensitive": val_insensitive,
    "Tuned hybrid": best_val_predictions
}

validation_scores = {
    name: calculate_metrics(predictions, val_gold)
    for name, predictions in validation_methods.items()
}

selected_method = max(
    validation_scores,
    key=lambda name: (
        validation_scores[name]["F1"],
        validation_scores[name]["Precision"],
        validation_scores[name]["Recall"]
    )
)

print("\nVALIDATION COMPARISON")
print("-" * 88)
print(
    f"{'Method':<22}{'Detected':>10}{'TP':>8}{'FP':>8}"
    f"{'FN':>8}{'Precision':>12}{'Recall':>10}{'F1':>10}"
)

for name, metrics in validation_scores.items():
    print(
        f"{name:<22}{metrics['Detected']:>10}"
        f"{metrics['TP']:>8}{metrics['FP']:>8}{metrics['FN']:>8}"
        f"{metrics['Precision']:>11.4f}"
        f"{metrics['Recall']:>10.4f}"
        f"{metrics['F1']:>10.4f}"
    )

print("\nSelected using validation F1:", selected_method)


VALIDATION COMPARISON
----------------------------------------------------------------------------------------
Method                  Detected      TP      FP      FN   Precision    Recall        F1
Case-sensitive              4204    3148    1056    1872     0.7488    0.6271    0.6826
Case-insensitive            7182    3200    3982    1820     0.4456    0.6375    0.5245
Tuned hybrid                3500    2755     745    2265     0.7871    0.5488    0.6467

Selected using validation F1: Case-sensitive


In [32]:
# ============================================================
# 11. FINAL EVALUATION ON THE SAME 200 TEST SENTENCES
# ============================================================

test_sensitive = [
    greedy_predict(tokens, True) for tokens in test_tokens
]

test_insensitive = [
    greedy_predict(tokens, False) for tokens in test_tokens
]

test_hybrid = [
    hybrid_predict(tokens, candidates, best_params)
    for tokens, candidates in zip(test_tokens, test_candidates)
]

test_methods = {
    "Case-sensitive": test_sensitive,
    "Case-insensitive": test_insensitive,
    "Tuned hybrid": test_hybrid
}

test_scores = {
    name: calculate_metrics(predictions, test_gold)
    for name, predictions in test_methods.items()
}

print("\nFINAL TEST RESULTS: 200 SENTENCES")
print("-" * 88)
print(
    f"{'Method':<22}{'Detected':>10}{'TP':>8}{'FP':>8}"
    f"{'FN':>8}{'Precision':>12}{'Recall':>10}{'F1':>10}"
)

for name, metrics in test_scores.items():
    print(
        f"{name:<22}{metrics['Detected']:>10}"
        f"{metrics['TP']:>8}{metrics['FP']:>8}{metrics['FN']:>8}"
        f"{metrics['Precision']:>11.4f}"
        f"{metrics['Recall']:>10.4f}"
        f"{metrics['F1']:>10.4f}"
    )

# Use the method selected on validation, not the test results.
final_predictions = test_methods[selected_method]

print("\nFINAL SELECTED METHOD:", selected_method)
print("F1-score:", round(test_scores[selected_method]["F1"], 4))


FINAL TEST RESULTS: 200 SENTENCES
----------------------------------------------------------------------------------------
Method                  Detected      TP      FP      FN   Precision    Recall        F1
Case-sensitive               204     157      47     215     0.7696    0.4220    0.5451
Case-insensitive             321     163     158     209     0.5078    0.4382    0.4704
Tuned hybrid                 175     148      27     224     0.8457    0.3978    0.5411

FINAL SELECTED METHOD: Case-sensitive
F1-score: 0.5451


In [33]:
# ============================================================
# 12. SHOW INCORRECT PREDICTIONS
# ============================================================

errors = []

for sentence_number, (predictions, gold, tokens) in enumerate(
    zip(final_predictions, test_gold, test_tokens), start=1
):
    pred_lookup = {
        (start, end, entity_type): phrase
        for start, end, entity_type, phrase in predictions
    }

    gold_entities = extract_entities(
        tokens, test_data[sentence_number - 1]["ner_tags"]
    )

    gold_lookup = {
        (start, end, entity_type): phrase
        for start, end, entity_type, phrase in gold_entities
    }

    predicted_set = set(pred_lookup)
    false_positives = predicted_set - gold
    false_negatives = gold - predicted_set

    for key in sorted(false_positives):
        errors.append((
            sentence_number,
            "FALSE POSITIVE",
            pred_lookup[key],
            key[2],
            " ".join(tokens)
        ))

    for key in sorted(false_negatives):
        errors.append((
            sentence_number,
            "FALSE NEGATIVE",
            gold_lookup[key],
            key[2],
            " ".join(tokens)
        ))

print("\nINCORRECT PREDICTIONS")
print("=" * 90)

for sentence_number, error_type, phrase, entity_type, text in errors[:10]:
    print(
        f"\nSentence {sentence_number} | {error_type} | "
        f"{phrase} [{entity_type}]"
    )
    print("Text:", text)

print(f"\nIncorrect predictions displayed: {min(10, len(errors))}")
print(f"Total incorrect predictions (FP + FN): {len(errors)}")

if len(errors) < 5:
    print("Fewer than five errors occurred for the selected method.")


INCORRECT PREDICTIONS

Sentence 1 | FALSE NEGATIVE | JAPAN [LOCATION]
Text: SOCCER - JAPAN GET LUCKY WIN , CHINA IN SURPRISE DEFEAT .

Sentence 1 | FALSE NEGATIVE | CHINA [PERSON]
Text: SOCCER - JAPAN GET LUCKY WIN , CHINA IN SURPRISE DEFEAT .

Sentence 2 | FALSE NEGATIVE | Nadim Ladki [PERSON]
Text: Nadim Ladki

Sentence 3 | FALSE NEGATIVE | AL-AIN [LOCATION]
Text: AL-AIN , United Arab Emirates 1996-12-06

Sentence 5 | FALSE NEGATIVE | Uzbekistan [LOCATION]
Text: But China saw their luck desert them in the second match of the group , crashing to a surprise 2-0 defeat to newcomers Uzbekistan .

Sentence 6 | FALSE NEGATIVE | Igor Shkvyrin [PERSON]
Text: China controlled most of the match and saw several chances missed until the 78th minute when Uzbek striker Igor Shkvyrin took advantage of a misdirected defensive header to lob the ball over the advancing Chinese keeper and into an empty net .

Sentence 7 | FALSE NEGATIVE | Oleg Shatskiku [PERSON]
Text: Oleg Shatskiku made sure of the w